<a href="https://colab.research.google.com/github/Abejirin-King/Implementing-CNN-with-tensorflow/blob/main/Abejirin_King_CNN_Exploration_Activity_Student_Improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CNN Exploration Lab
### From a Basic CNN to Regularization and Grad-CAM

**Estimated time:** 60 minutes  
**Framework:** TensorFlow / Keras  
**Dataset:** Fashion-MNIST

---

## Learning goals

By the end of this activity, you should be able to:

- build an image pipeline with `tf.data`;
- construct a simple Convolutional Neural Network;
- explain the roles of convolution, padding, pooling, and the classifier head;
- use image augmentation to make a CNN more robust to small transformations;
- explain where Batch Normalization fits in a CNN;
- combine Batch Normalization, Dropout, and L2 regularization;
- inspect intermediate CNN feature maps;
- use Grad-CAM to identify image regions that influence a prediction.

### Before you begin

Watch this short video before starting the activity:

▶️ **CNN video:** https://www.youtube.com/watch?v=7womw-TI6Ss

You do **not** need to memorize everything in the video. While watching, focus on this question:

> **How does a CNN gradually transform an image into information that can be used for classification?**

---

## How to use this notebook

This is a guided activity, not a copy-and-run notebook.

Whenever you see `your answer`:

1. read the explanation immediately above the code;
2. use the hint or linked documentation if needed;
3. replace `your answer` with your answer;
4. run the cell and inspect the result;
5. if the output is unexpected, use the error or visualization to revise your answer.

The goal is to understand **why each component is being used**, not just to make every cell execute.


## 0. Setup

Run the cell below. You do not need to change anything here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers, regularizers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

# Part 1 — Meet the dataset

We will use **Fashion-MNIST**, a small image dataset included with Keras.

It contains grayscale images of clothing items such as shoes, coats, shirts, bags, and dresses.

Each image has shape:

`28 × 28`

and belongs to one of **10 classes**.

Because the dataset is available directly through Keras, it is quick to download and works well for a short Colab activity.

**Resource:**  
https://www.tensorflow.org/tutorials/keras/classification

In [ ]:
fashion_mnist = tf.keras.datasets.fashion_mnist

(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print("Training images:", train_images.shape)
print("Training labels:", train_labels.shape)
print("Test images:", test_images.shape)

### 1.1 Inspect the data

Before training a model, we need to understand what the dataset actually looks like.

#### What you are trying to do

Display **12 training images** in a grid and show the correct class name above each image.

This helps you check:

- what the images look like;
- how the labels are represented;
- whether some classes look visually similar.

#### How to approach it

The training images and labels use the **same index**.

For example:

```python
train_images[0]
```

is the image whose label is stored in:

```python
train_labels[0]
```

The label itself is an integer such as `9`. To turn that integer into a readable class name, use it as an index into `class_names`:

```python
class_names[train_labels[0]]
```

To display several images in one figure, use `plt.subplot(rows, columns, position)`.

#### Your task

Complete the three `required values` values so that:

- the loop displays **12 images**;
- the images are arranged in a **3 × 4** grid;
- each title shows the correct class name.

**Resource:**  
https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplot.html

In [ ]:
plt.figure(figsize=(10, 6))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(train_images[i], cmap="gray")
    plt.title(class_names[train_labels[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()

### Pause and observe

Do not rush into the CNN yet. Use the images you just plotted to form an expectation about the problem.

#### What you are trying to do

Reason about the dataset **before** seeing model results.

Think about shape, similarity between classes, and object position.

#### Your task

Answer briefly:

1. Which classes look visually easy to distinguish, and why?
2. Which classes do you expect a CNN to confuse?
3. Are the objects always in exactly the same position?

**Your observations:**

`Trouser, sneaker, bag, and ankle boot are generally easier to distinguish because their overall shapes are more distinctive. Shirts, T-shirts, pullovers, and coats can be harder to separate because their silhouettes overlap. The objects are not always in exactly the same position, which is one reason mild translation augmentation can help.`

# Part 2 — Build a `tf.data` pipeline

The arrays returned by Fashion-MNIST are small enough to fit in memory, but we will still use `tf.data`.

This gives us a workflow that scales to larger machine-learning projects.

We will:

1. create a validation split;
2. convert arrays to `tf.data.Dataset` objects;
3. normalize the pixels;
4. shuffle the training data;
5. batch the data;
6. prefetch batches.

The **test set remains untouched** until final evaluation.

**Resource:**  
https://www.tensorflow.org/guide/data

In [ ]:
# Create a validation split from the original training data.
val_images = train_images[-5000:]
val_labels = train_labels[-5000:]

train_images_small = train_images[:-5000]
train_labels_small = train_labels[:-5000]

print(train_images_small.shape, val_images.shape, test_images.shape)

### 2.1 Preprocess each image

Neural networks work better when the input data is represented consistently.

#### What you are trying to do

Create a preprocessing function that transforms each Fashion-MNIST image from:

```text
shape: (28, 28)
pixel values: 0 ... 255
```

into:

```text
shape: (28, 28, 1)
pixel values: approximately 0 ... 1
```

#### How to approach it

There are two steps.

**Step 1: Normalize the pixel values**

The images are stored as integer pixel values from `0` to `255`. Convert them to floating-point values, then divide by `255.0`.

TensorFlow provides:

```python
tf.cast(...)
```

for changing the datatype.

**Step 2: Add the channel dimension**

A `Conv2D` layer expects each image to include a channel dimension.

Fashion-MNIST is grayscale, so it has **one channel**:

```text
(28, 28) → (28, 28, 1)
```

Use:

```python
tf.expand_dims(...)
```

and add the new dimension at the end.

#### Your task

Complete the datatype, normalization value, and axis below.

**Resources:**  
- `tf.cast`: https://www.tensorflow.org/api_docs/python/tf/cast  
- `tf.expand_dims`: https://www.tensorflow.org/api_docs/python/tf/expand_dims

In [ ]:
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.expand_dims(image, axis=-1)
    return image, label

### 2.2 Build the `tf.data` pipelines

We now have a preprocessing function. The next step is to decide **how examples will be delivered to the model during training**.

#### What you are trying to do

Build three input pipelines:

- `train_ds`
- `val_ds`
- `test_ds`

All three datasets should:

1. preprocess the images;
2. group examples into batches;
3. prefetch the next batch while the model is working.

The training dataset needs one additional step: **shuffling**.

#### How to approach it

A typical training pipeline follows this order:

```text
raw examples
    ↓
map(preprocessing)
    ↓
shuffle
    ↓
batch
    ↓
prefetch
```

Why only shuffle the training set?

During training, we do not want the model to repeatedly see examples in the same order. Validation and test data do not need this because they are only used for evaluation.

Use:

```python
.shuffle(buffer_size, seed=SEED)
.batch(BATCH_SIZE)
```

For the validation and test datasets, you only need the batching operation after `.map(...)`.

`prefetch(tf.data.AUTOTUNE)` has already been provided.

#### Your task

Complete the missing `shuffle` and `batch` operations.

**Resources:**  
- Shuffle: https://www.tensorflow.org/api_docs/python/tf/data/Dataset#shuffle  
- Batch: https://www.tensorflow.org/api_docs/python/tf/data/Dataset#batch  
- Prefetch: https://www.tensorflow.org/api_docs/python/tf/data/Dataset#prefetch

In [ ]:
BATCH_SIZE = 128

train_ds = tf.data.Dataset.from_tensor_slices(
    (train_images_small, train_labels_small)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_images, val_labels)
)

test_ds = tf.data.Dataset.from_tensor_slices(
    (test_images, test_labels)
)

train_ds = (
    train_ds
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(10000, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    test_ds
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Part 3 — Build a baseline CNN

A CNN has two broad jobs:

### Feature extractor
Convolution and pooling layers transform the image into useful visual features.

### Classifier head
The learned features are converted into class probabilities.

For this activity, the model will follow this structure:

**Input → Convolution → Pooling → Convolution → Pooling → Global Average Pooling → Dense classifier → Prediction**

Notice that **Global Average Pooling is not the classifier**.

It reduces each feature map to a compact feature representation.  
The **Dense layers after it perform the classification**.

### 3.1 Build the baseline CNN

Now that the data pipeline is ready, we can build our first CNN.

#### What you are trying to do

Create a small model with two clearly separated parts:

**1. Feature extractor**

```text
Convolution → Max Pooling → Convolution → Max Pooling
```

The convolutional layers learn visual patterns. Pooling reduces the spatial size of the feature maps.

**2. Classifier head**

```text
Global Average Pooling → Dense → Dense prediction layer
```

`GlobalAveragePooling2D` is **not** the classifier. It converts the final set of feature maps into a compact feature vector. The Dense layers then use that vector to classify the image.

#### How to approach it

Build the model using these settings:

| Layer | Setting |
|---|---|
| Conv 1 | 32 filters, `3 × 3`, `same` padding |
| Max Pool 1 | `2 × 2` |
| Conv 2 | 64 filters, `3 × 3`, `same` padding |
| Max Pool 2 | `2 × 2` |
| Global Average Pooling | no parameters needed |
| Hidden Dense | 64 neurons, ReLU |
| Output Dense | 10 neurons, softmax |

Remember:

```python
filters=32
```

means that the convolution learns **32 different filters**, producing 32 feature maps.

```python
kernel_size=(3, 3)
```

means that each filter looks at a `3 × 3` local region at a time.

```python
padding="same"
```

keeps the height and width unchanged when the stride is 1.

#### Your task

Complete the missing filter counts, kernel sizes, padding choices, pooling sizes, and Dense layer sizes.

**Resources:**  
- Conv2D: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D  
- MaxPooling2D: https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPooling2D  
- GlobalAveragePooling2D: https://www.tensorflow.org/api_docs/python/tf/keras/layers/GlobalAveragePooling2D  
- Dense: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense

In [ ]:
baseline_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),

    # -------- Feature extractor --------
    layers.Conv2D(
        filters=32,
        kernel_size=(3, 3),
        padding="same",
        activation="relu",
        name="baseline_conv_1"
    ),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Conv2D(
        filters=64,
        kernel_size=(3, 3),
        padding="same",
        activation="relu",
        name="baseline_conv_2"
    ),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.GlobalAveragePooling2D(),

    # -------- Classifier head --------
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

baseline_model.summary()

### 3.2 Visualize the architecture

Reading `model.summary()` is useful, but a diagram can make the flow through the network easier to understand.

#### What you are trying to do

Ask Keras to generate a diagram of the model showing:

- each layer;
- the layer names;
- the tensor shapes flowing between layers.

#### How to approach it

Keras provides a utility specifically for this purpose under:

```python
tf.keras.utils
```

The function accepts the model as its first argument. The other arguments in the cell are already configured for you.

#### Your task

Use the documentation to identify the missing function name.

**Resource:**  
https://www.tensorflow.org/api_docs/python/tf/keras/utils/plot_model

In [ ]:
tf.keras.utils.plot_model(
    baseline_model,
    show_shapes=True,
    show_layer_names=True,
    rankdir="LR"
)

### Checkpoint — CNN fundamentals

Before training, make sure you can explain what the architecture is doing.

#### How to approach these questions

Do not answer with only definitions. Relate each answer to the model you just built.

**1. What does the number of filters in `Conv2D` control?**  
`The number of filters controls how many different feature maps the convolutional layer produces, allowing it to learn different visual patterns.`

**2. What does `padding="same"` try to preserve?**  
`Same padding is intended to preserve the input height and width when the convolution uses stride 1.`

**3. What does max pooling do to the spatial representation?**  
`Max pooling reduces the height and width of feature maps while retaining the strongest responses in local regions.`

**4. Which layers make up the classifier head in this network? What does Global Average Pooling do before them?**  
`The classifier head here is Global Average Pooling followed by Dense(64, ReLU) and Dense(10, softmax). Global Average Pooling summarizes each feature map into one value before the Dense layers perform classification.`

**5. If a convolutional filter learns to detect an edge, why can the same filter detect that edge in different parts of the image?**  
`A convolution filter is shared across the image, so the same learned edge detector is applied at every spatial location.`

### 3.3 Train the baseline

Run the next cell.

We only train for a few epochs because the goal is to **experiment**, not to maximize benchmark accuracy.

In [ ]:
baseline_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

baseline_history = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)

# Part 4 — Translation robustness and augmentation

A convolutional filter is applied across the image, which lets the same learned feature be detected at different positions.

However, a CNN is **not automatically perfectly invariant** to every shift, rotation, or change in scale.

One way to improve robustness is **data augmentation**.

During training, augmentation generates slightly modified versions of the images while keeping the class label unchanged.

For this dataset we will experiment with:

- small translations;
- small rotations;
- small zooms.

We intentionally keep these transformations mild because Fashion-MNIST images are small.

**Resource:**  
https://www.tensorflow.org/tutorials/images/data_augmentation

### 4.1 Build an augmentation pipeline

A CNN can detect a learned feature at different spatial positions, but that does **not** mean it will automatically handle every shift, rotation, or change in scale equally well.

Data augmentation lets us expose the model to realistic variations during training.

#### What you are trying to do

Create a small augmentation pipeline that randomly applies:

1. translation;
2. rotation;
3. zoom.

The class label should still remain correct after the transformation.

#### How to approach it

Keras provides preprocessing layers for these transformations:

```python
layers.RandomTranslation(...)
layers.RandomRotation(...)
layers.RandomZoom(...)
```

The transformation strengths have already been chosen for you because Fashion-MNIST images are only `28 × 28`. We want **small** changes, not transformations that destroy the object.

The first layer receives separate horizontal and vertical translation factors. Rotation and zoom each receive one factor here.

#### Your task

Replace each `the corresponding augmentation layer` with the correct Keras augmentation layer name.

**Resource:**  
https://www.tensorflow.org/tutorials/images/data_augmentation

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomTranslation(
        height_factor=0.08,
        width_factor=0.08
    ),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.08),
], name="augmentation")

### 4.2 Visualize the augmentation

Before using augmentation during training, inspect what it actually does to an image.

#### What you are trying to do

Take one Fashion-MNIST image and pass it through the augmentation pipeline **nine different times**.

Because the transformations are random, each output should look slightly different.

#### How to approach it

A Keras preprocessing pipeline can be called just like a model:

```python
some_layer_or_model(input_tensor, training=True)
```

We use `training=True` because random augmentation should be active during training.

The plotting code is already provided. You only need to send `sample_image` through the augmentation pipeline you created above.

#### Your task

Replace `data_augmentation` with the name of your augmentation pipeline.

In [ ]:
sample_image = train_images_small[0].astype("float32") / 255.0
sample_image = np.expand_dims(sample_image, axis=-1)
sample_image = np.expand_dims(sample_image, axis=0)

plt.figure(figsize=(9, 6))

for i in range(9):
    augmented = data_augmentation(sample_image, training=True)

    plt.subplot(3, 3, i + 1)
    plt.imshow(augmented[0, :, :, 0], cmap="gray")
    plt.axis("off")

plt.tight_layout()
plt.show()

### Think about the augmentation choices

Augmentation is useful only when the transformed image still represents the same class.

#### What you are trying to reason about

Fashion-MNIST images are very small and mostly centered. A transformation that is too strong could rotate a garment unnaturally or move much of it outside the image.

#### Your task

Explain why **very large** rotations or translations could hurt rather than help this model.

`Very large rotations or translations can make clothing images unrealistic, move important pixels outside the frame, or change the visual structure enough that the original label is no longer reliable. That can teach the model examples that do not represent the real data distribution.`

# Part 5 — Batch Normalization

So far our convolutional blocks have been:

```text
Convolution → ReLU → Pooling
```

We will now introduce **Batch Normalization**.

### What problem are we trying to address?

During training, the values produced inside a neural network can change as the weights are updated. In deeper networks, this can make optimization harder.

Batch Normalization normalizes intermediate activations using statistics from the current mini-batch during training. It also learns parameters that allow the network to rescale and shift those normalized values.

In practice, it often makes optimization more stable.

For this activity, we will use the pattern:

```text
Convolution → Batch Normalization → ReLU → Pooling
```

Notice that Batch Normalization is placed **before the ReLU activation** in the architecture we are building.

**Resource:**  
https://www.tensorflow.org/api_docs/python/tf/keras/layers/BatchNormalization

### 5.1 Think before coding

#### What you are trying to understand

We want this order:

```text
Convolution → Batch Normalization → ReLU
```

But earlier we wrote:

```python
layers.Conv2D(..., activation="relu")
```

If ReLU is already inside the `Conv2D` layer, it happens immediately after the convolution.

#### How to reason about it

To insert another layer **between** the convolution and ReLU, we need to separate the activation from the convolution.

Think about what the architecture should look like in Keras when each operation is its own layer.

#### Your task

In one or two sentences, explain why we should remove `activation="relu"` from `Conv2D` when Batch Normalization must come before ReLU.

`ReLU must be a separate layer so Batch Normalization can operate on the raw convolution output first. If ReLU remains inside Conv2D, the activation happens before Batch Normalization and the intended Conv → BatchNorm → ReLU order is lost.`

# Part 6 — Build a stronger CNN

Now we will combine several ideas:

- data augmentation;
- convolution;
- Batch Normalization;
- max pooling;
- L2 regularization;
- Dropout;
- an explicit classifier head.

These techniques do **different jobs**.

| Technique | Main purpose |
|---|---|
| Data augmentation | Expose the model to realistic variations |
| Batch Normalization | Stabilize activations and optimization |
| L2 | Discourage excessively large weights |
| Dropout | Reduce reliance on specific activations |

Do not think of them as interchangeable.

### 6.1 Build the improved CNN

We are now going to combine the ideas introduced so far into one model.

#### What you are trying to do

Build a CNN with:

- augmentation at the input;
- two convolutional blocks;
- Batch Normalization inside each block;
- L2 regularization on trainable kernels;
- Global Average Pooling;
- a Dense classifier head;
- Dropout before the final prediction layer.

The architecture should look like:

```text
Input
  ↓
Data augmentation
  ↓
Conv → BatchNorm → ReLU → MaxPool
  ↓
Conv → BatchNorm → ReLU → MaxPool
  ↓
Global Average Pooling
  ↓
Dense classifier
  ↓
Dropout
  ↓
10-class prediction
```

#### How to approach it

Use the following values:

| Component | Value |
|---|---|
| First convolution | 32 filters |
| Second convolution | 64 filters |
| L2 strength | `1e-4` |
| Hidden Dense layer | 64 neurons |
| Dropout | `0.30` |
| Output layer | 10 neurons |

For Batch Normalization, insert:

```python
layers.BatchNormalization()
```

after each convolution.

For L2 regularization, use:

```python
regularizers.l2(1e-4)
```

The convolutional layers already have `use_bias=False`. This is common when Batch Normalization immediately follows the convolution because Batch Normalization already learns a shift parameter.

#### Your task

Complete the filter counts, L2 values, Batch Normalization layers, Dense size, Dropout rate, and output size.

**Resources:**  
- Batch Normalization: https://www.tensorflow.org/api_docs/python/tf/keras/layers/BatchNormalization  
- Dropout: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout  
- L2: https://www.tensorflow.org/api_docs/python/tf/keras/regularizers/L2

In [ ]:
improved_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    data_augmentation,

    # -------- Feature extractor --------
    layers.Conv2D(
        32,
        3,
        padding="same",
        use_bias=False,
        kernel_regularizer=regularizers.l2(1e-4),
        name="conv_1"
    ),
    layers.BatchNormalization(),
    layers.Activation("relu", name="conv_1_relu"),
    layers.MaxPooling2D(2),

    layers.Conv2D(
        64,
        3,
        padding="same",
        use_bias=False,
        kernel_regularizer=regularizers.l2(1e-4),
        name="conv_2"
    ),
    layers.BatchNormalization(),
    layers.Activation("relu", name="conv_2_relu"),
    layers.MaxPooling2D(2),

    layers.GlobalAveragePooling2D(name="global_average_pooling"),

    # -------- Classifier head --------
    layers.Dense(
        64,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name="classifier_dense"
    ),
    layers.Dropout(0.30),
    layers.Dense(10, activation="softmax", name="predictions")
])

improved_model.summary()

### 6.2 Train the improved model

In [ ]:
improved_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

improved_history = improved_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)

### 6.3 Compare training and validation accuracy

Training a model gives us more information than just the final accuracy.

#### What you are trying to do

Plot two curves:

- training accuracy;
- validation accuracy.

Comparing them helps you see whether the model is improving similarly on data it trains on and data it does not train on.

A large gap between the two can be a sign of overfitting.

#### How to approach it

The object returned by `model.fit(...)` contains a dictionary called:

```python
history.history
```

Because we compiled the model with:

```python
metrics=["accuracy"]
```

Keras records keys for:

```text
accuracy
val_accuracy
```

The first refers to the training set. The second refers to the validation set.

#### Your task

Use the correct dictionary keys in the two `the appropriate history keys (`accuracy` and `val_accuracy`)` positions.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    improved_history.history["accuracy"],
    label="Training accuracy"
)

plt.plot(
    improved_history.history["val_accuracy"],
    label="Validation accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

### Reflect on the comparison

We changed several things between the baseline model and the improved model:

- augmentation;
- Batch Normalization;
- L2 regularization;
- Dropout.

#### What you are trying to understand

If the improved model performs differently, we cannot immediately claim that one specific technique caused the difference because several variables changed together.

#### Your task

**1. Do these results prove that every added technique helped individually? Explain.**  
`No. Because augmentation, Batch Normalization, L2 regularization, and Dropout were changed together, the comparison shows the combined effect of the changes rather than the individual contribution of each technique.`

**2. Suppose you wanted to measure only the effect of Batch Normalization. What would you keep fixed, and what would you change?**  
`Keep the dataset split, preprocessing, architecture, optimizer, learning rate, batch size, number of epochs, random seed, and all other hyperparameters fixed. Change only the presence or configuration of Batch Normalization.`

### 6.4 Final evaluation on the untouched test set

Until now, we used the **validation set** while developing the model.

Now that the architecture is fixed, evaluate the improved model **once** on the test set.

This gives us a final estimate on data that was not used to choose the architecture.

In [ ]:
test_loss, test_accuracy = improved_model.evaluate(test_ds, verbose=0)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

# Part 7 — What has the CNN learned?

A CNN does not jump directly from pixels to a class label.

Each convolutional layer produces **feature maps**.

Early layers often respond to relatively simple structures.  
Deeper layers combine earlier information into more task-specific patterns.

Let's inspect those internal activations.

### 7.1 Create a feature extractor

Our trained model normally returns only the final class probabilities. But we can create another model that stops at an intermediate layer.

#### What you are trying to do

Create a new Keras model that:

- receives the **same image input** as the trained CNN;
- returns the activations produced by the first convolutional block.

This lets us inspect the individual **feature maps** generated by that layer.

#### How to approach it

We named the activation after the first convolution:

```text
conv_1_relu
```

You can retrieve a named layer using:

```python
model.get_layer("layer_name")
```

Then create a new model whose output is:

```python
target_layer.output
```

The input should still be the input of the original model.

#### Your task

1. Fill in the name of the layer you want to inspect.
2. Fill in the tensor that should become the output of `feature_extractor`.

**Resource:**  
https://keras.io/guides/functional_api/#extract-and-reuse-nodes-in-the-graph-of-layers

In [ ]:
target_layer = improved_model.get_layer("conv_1_relu")

feature_extractor = tf.keras.Model(
    inputs=improved_model.inputs,
    outputs=target_layer.output
)

### 7.2 Visualize the feature maps

Now use the feature extractor on a real test image.

#### What you are trying to do

Choose one image from the test set, send it through the feature extractor, and visualize up to **16 channels** from the resulting activation tensor.

Each channel is one learned feature map.

#### How to approach it

The test set is an array, so an image can be selected with an integer index:

```python
test_images[0]
test_images[12]
test_images[250]
```

Any valid index is acceptable.

The rest of the code:

1. normalizes the image;
2. adds batch and channel dimensions;
3. sends the image through the feature extractor;
4. plots individual channels.

#### Your task

Replace `the selected test-image index` with the index of any test image you want to investigate.

In [ ]:
example_index = 0

image = test_images[example_index].astype("float32") / 255.0
image_input = image[np.newaxis, ..., np.newaxis]

feature_maps = feature_extractor.predict(image_input, verbose=0)

plt.figure(figsize=(10, 8))

for channel in range(min(16, feature_maps.shape[-1])):
    plt.subplot(4, 4, channel + 1)
    plt.imshow(feature_maps[0, :, :, channel], cmap="viridis")
    plt.axis("off")

plt.suptitle("Feature maps from the first convolutional block")
plt.tight_layout()
plt.show()

### Observe the learned feature maps

There is no requirement that every feature map has an obvious human-readable meaning.

#### What you are trying to do

Compare several channels and describe what appears different about their responses.

You might notice that some channels emphasize:

- boundaries;
- particular parts of the silhouette;
- bright or dark regions;
- very little at all.

#### Your task

Choose at least two feature maps and describe what each one appears to respond to.

`Different channels can respond to different visual structures. For example, one channel may emphasize edges or the outer silhouette while another may respond more strongly to bright or dark regions or particular parts of the garment.`

Then answer:

**Why should we be cautious about claiming that a particular filter has one exact human-readable meaning?**

`A filter's response depends on the learned weights, the input, and interactions with other layers, so a visual pattern in one feature map does not prove that the filter has one single human-readable meaning.`

# Part 8 — Grad-CAM

Feature maps show us what happens inside a layer.

Grad-CAM asks a different question:

> **Which spatial regions of the image were important for a particular prediction?**

Grad-CAM combines:

1. the spatial feature maps from a convolutional layer;
2. gradients of the selected class score with respect to those feature maps.

We will use the final convolutional activation, `conv_2_relu`.

### Reference

Keras Grad-CAM example:  
https://keras.io/examples/vision/grad_cam/

### 8.1 Choose an image and inspect the prediction

Before creating a Grad-CAM heatmap, we first need a prediction to explain.

#### What you are trying to do

Choose one test image and record:

- its true class;
- the class predicted by the CNN;
- the model's confidence in that prediction.

Grad-CAM will then help us investigate which spatial regions influenced that particular class prediction.

#### How to approach it

Just as in the feature-map activity, choose any valid test-set index:

```python
test_images[index]
```

It is useful to start with a correctly classified image. Later, you can repeat the activity with a misclassified example.

The provided code will handle normalization, prediction, and visualization.

#### Your task

Replace `the selected test-image index` with the index of the test image you want Grad-CAM to explain.

In [ ]:
gradcam_index = 0

image = test_images[gradcam_index].astype("float32") / 255.0
image_input = image[np.newaxis, ..., np.newaxis]

predictions = improved_model.predict(image_input, verbose=0)

predicted_class = np.argmax(predictions[0])
confidence = predictions[0][predicted_class]

print("True class:", class_names[test_labels[gradcam_index]])
print("Predicted class:", class_names[predicted_class])
print("Confidence:", float(confidence))

plt.imshow(image, cmap="gray")
plt.axis("off")
plt.show()

### 8.2 Build a model for Grad-CAM

Grad-CAM needs access to **two outputs at the same time**.

#### What you are trying to do

Create a helper model that returns:

1. the spatial activations from the final convolutional block;
2. the final class probabilities from the CNN.

Why both?

The convolutional activations tell us **where visual information exists spatially**.  
The final prediction tells us **which class score we want to explain**.

Grad-CAM uses gradients to connect those two pieces.

#### How to approach it

The final convolutional activation layer has already been named:

```python
last_conv_layer_name = "conv_2_relu"
```

Retrieve its output using:

```python
improved_model.get_layer(last_conv_layer_name).output
```

The original model's final output is available from:

```python
improved_model.outputs[0]
```

#### Your task

Complete the two outputs of `grad_model`:

- the last convolutional activation;
- the final prediction tensor.

In [ ]:
last_conv_layer_name = "conv_2_relu"

grad_model = tf.keras.Model(
    inputs=improved_model.inputs,
    outputs=[
        improved_model.get_layer(last_conv_layer_name).output,
        improved_model.outputs[0]
    ]
)

### 8.3 Compute the Grad-CAM heatmap

Read the code before running it.

Try to identify:

- where the class score is selected;
- where the gradient is calculated;
- where each feature map receives an importance weight.

In [ ]:
def make_gradcam_heatmap(img_array, grad_model, class_index=None):
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)

        if class_index is None:
            class_index = tf.argmax(predictions[0])

        class_score = predictions[:, class_index]

    grads = tape.gradient(class_score, conv_outputs)

    # Average the gradient across spatial dimensions.
    channel_weights = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weight each feature map by its importance.
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(
        conv_outputs * channel_weights,
        axis=-1
    )

    # Keep positive contributions and normalize.
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy()


heatmap = make_gradcam_heatmap(
    image_input,
    grad_model,
    predicted_class
)

plt.imshow(heatmap, cmap="jet")
plt.axis("off")
plt.title("Grad-CAM heatmap")
plt.show()

### 8.4 Overlay Grad-CAM on the original image

A heatmap by itself is useful, but it is easier to interpret when it is placed on top of the original image.

#### What you are trying to do

Overlay the Grad-CAM heatmap on the Fashion-MNIST image while still keeping the original clothing item visible.

#### How to approach it

The final convolutional feature map is smaller than the original `28 × 28` image, so the code first resizes the heatmap.

Matplotlib's `imshow(...)` supports a parameter that controls **transparency**:

```text
0.0 = completely transparent
1.0 = completely opaque
```

A value somewhere around `0.4` to `0.5` usually makes both the image and heatmap visible.

#### Your task

Find the Matplotlib argument that controls transparency and assign it a reasonable value between `0` and `1`.

**Resource:**  
https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html

In [ ]:
# The convolutional heatmap is smaller than the 28×28 input.
# Resize it before overlaying it on the original image.
heatmap_resized = tf.image.resize(
    heatmap[..., np.newaxis],
    size=(28, 28),
    method="bilinear"
).numpy().squeeze()

plt.figure(figsize=(5, 5))

plt.imshow(image, cmap="gray")
plt.imshow(
    heatmap_resized,
    cmap="jet",
    alpha=0.45,
    interpolation="bilinear"
)

plt.title(
    f"Prediction: {class_names[predicted_class]} "
    f"({confidence:.2%})"
)
plt.axis("off")
plt.show()

## Grad-CAM discussion

The heatmap is useful only if you interpret it critically.

#### What you are trying to do

Use Grad-CAM as evidence about **where the model was sensitive for this prediction**, not as proof that you have completely explained the neural network.

#### Your task

**1. Which region of the image received the strongest Grad-CAM response?**  
`The strongest response should appear over the most informative part of the clothing item rather than the empty background; the exact location should be confirmed from the heatmap produced for the selected image.`

**2. Does that region make sense for the predicted clothing class? Explain.**  
`If the model is using a sensible feature, the highlighted region should overlap a discriminative part of the garment, such as its silhouette or distinctive structure. If the heatmap emphasizes irrelevant background regions, that may indicate a less reliable explanation.`

**3. Repeat Grad-CAM with one correctly classified image and one misclassified image. What differences do you notice?**  
`A correctly classified image will often show concentrated attention on useful garment structure, while a misclassified image may show attention on ambiguous or less relevant regions. The exact difference depends on the two images and the trained model.`

**4. Why does a Grad-CAM heatmap not give a complete explanation of how the CNN reasons?**  
`Grad-CAM summarizes spatial importance for one chosen class and layer. It does not expose every internal computation, feature interaction, or decision made throughout the entire network.`

# Final challenge

Choose **one** experiment. The goal is to change one thing, observe the result, and explain what happened.

### Option A — Augmentation

Change **one** augmentation strength, such as rotation or translation.

Before running it, predict what you expect to happen. Then retrain briefly and compare.

**Prediction and observation:**  
`Option A: I would slightly increase the translation factor and expect validation performance to change only modestly because the model would see more positional variation. After retraining, I would compare the validation accuracy with the original run rather than assuming the change helped.`

### Option B — Dropout

Change the Dropout rate from `0.30` to another value.

Think first: would stronger Dropout always improve validation performance?

**Prediction and observation:**  
`Option B: Stronger Dropout does not always improve validation performance. I would expect a moderate rate to reduce overfitting, but too much Dropout can make learning harder and lower both training and validation accuracy.`

### Option C — Explainability

Run Grad-CAM on examples from **three different classes**.

Compare where the network focuses.

**Observation:**  
`Option C: Grad-CAM should highlight different spatial regions for different clothing classes because the model relies on class-specific visual patterns. The exact regions should be confirmed from the generated heatmaps.`

# Exit ticket

Use your own words. Aim for one or two clear sentences per question.

**1. What is the difference between the CNN's feature extractor and classifier head?**  
`The feature extractor learns visual patterns such as edges and shapes through convolutional layers, while the classifier head converts the extracted representation into class probabilities.`

**2. Why can data augmentation improve generalization?**  
`Augmentation exposes the model to realistic variations of the training images, which can reduce reliance on exact positions or appearances and improve performance on unseen examples.`

**3. Where did we place Batch Normalization in each convolutional block, and why did we separate ReLU from `Conv2D`?**  
`Batch Normalization was placed after each convolution and before ReLU. ReLU was separated from Conv2D so the order is explicitly Conv → BatchNorm → ReLU.`

**4. How are Dropout and L2 regularization different?**  
`Dropout randomly removes activations during training, while L2 regularization adds a penalty for large weights to the training objective.`

**5. What do intermediate feature maps let us inspect?**  
`Intermediate feature maps let us inspect the spatial patterns and visual features produced inside convolutional layers before the final classification.`

**6. What does Grad-CAM help us investigate?**  
`Grad-CAM helps investigate which spatial regions contributed most strongly to a selected class prediction.`

---

## Useful references

- Fashion-MNIST tutorial: https://www.tensorflow.org/tutorials/keras/classification
- `tf.data`: https://www.tensorflow.org/guide/data
- Conv2D: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D
- MaxPooling2D: https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPooling2D
- Data augmentation: https://www.tensorflow.org/tutorials/images/data_augmentation
- Batch Normalization: https://www.tensorflow.org/api_docs/python/tf/keras/layers/BatchNormalization
- Dropout: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout
- Regularizers: https://www.tensorflow.org/api_docs/python/tf/keras/regularizers
- Plotting a Keras model: https://www.tensorflow.org/api_docs/python/tf/keras/utils/plot_model
- Grad-CAM example: https://keras.io/examples/vision/grad_cam/